# FLAG 2027 — NB-3: targeted submissions from configs (no ablation)

Each entry of `RUNS` is trained on all 70 speakers (ensemble of 10) and scored on all four dev cells
-> one zip per run. Per-cell merging of zips that CodaBench has scored is done locally with
`kaggle/merge_cells.py` (no retraining).

Round 2 (after NB-2): the voice feature decides the Bangla cells, and internal validation cannot see Bangla.
So each run changes ONE thing vs `r2_ecapa192` (own ECAPA-192, same recipe as EXP-007):
* `r2_ecapa192_mix` - + duration-matched voice crops (Bangla-length) as training views
* `r2_wavlm`        - WavLM-Large layers 4-9 instead of ECAPA (internal 26.61 vs control 26.16)

**Settings:** GPU, Internet OFF. Inputs: `flag2027-feats-v2` + `flag2027-features`. ~10 min per run on a T4.

In [ ]:
import os, sys
from pathlib import Path
LOCAL = os.environ.get("FLAG_LOCAL") == "1"          # CPU smoke test on a mini copy of the data
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| working", WORKDIR)

In [ ]:
# Writes the helper modules next to this notebook (no .py upload needed).
import base64
open('flag_lib.py', 'wb').write(base64.b64decode(''.join('''
...
cmV0dXJuIHBhdGgK
'''.split())))
open('flag_v2.py', 'wb').write(base64.b64decode(''.join('''
...
'''.split())))
print('written:', ['flag_lib.py', 'flag_v2.py'])

In [ ]:
import numpy as np, pandas as pd, json
os.environ["FLAG_DATA"] = str(IN)
import flag_v2 as V
V.polarity_preflight()
store = V.FeatureStore(IN)

# ---- edit here ---------------------------------------------------------------------------------
REC = dict(V.BASE, head="mlp", drop=0.3, emb=128, epochs=40)         # = EXP-007 recipe
RUNS = {
    "r2_ecapa192":     dict(REC, voice="ecapa192"),
    "r2_ecapa192_mix": dict(REC, voice="ecapa192", voice_view="mix"),
    "r2_wavlm":        dict(REC, voice="wavlm_L4-9"),
}
CCA_W = 0.25
N_MODELS = 2 if LOCAL else 10
if LOCAL:
    RUNS = {k: dict(v, epochs=2) for k, v in RUNS.items()}
    RUNS = {k: v for k, v in RUNS.items() if store.has("voice", v["voice"])}
# -------------------------------------------------------------------------------------------------
for k, v in RUNS.items():
    print(k, {x: v[x] for x in ("face", "voice", "voice_view", "w_aam", "same_gender_only")})

In [ ]:
import time
SUBS = WORKDIR / "submissions"; SUBS.mkdir(exist_ok=True)
rows = []
for name, cfg in RUNS.items():
    out = SUBS / f"{name}.zip"
    if out.exists():
        print("skip (exists)", out.name); continue
    t0 = time.time()
    sc = V.dev_scores(store, cfg, n_models=N_MODELS, cca_w=CCA_W)
    np.savez(SUBS / f"{name}_scores.npz", **{"/".join(k): v for k, v in sc.items()})
    V.write_zip(store, out, sc); V.check_zip(store, out)
    rows.append(dict(zip=out.name, sec=round(time.time() - t0), cfg=json.dumps(cfg)))
    pd.DataFrame(rows).to_csv(SUBS / "runs.csv", mode="a", header=not (SUBS / "runs.csv").exists(), index=False)
    rows = []
    print(f"{name}: {round(time.time() - t0)}s")
print(sorted(p.name for p in SUBS.glob("*.zip")))